# Steam Dataset Crawler

This is the single orchestration notebook for both smoke-test runs and the full SLURM run.
All heavy lifting lives in `src/steam_crawler`, while this notebook controls runtime limits and stage execution.


In [12]:
# Resolve the project root no matter whether Jupyter starts in the repo root or the notebook directory.
from pathlib import Path
import sys

candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve() / "steam-crawler",
    Path.cwd().resolve().parent / "steam-crawler",
]
ROOT_DIR = next(path for path in candidates if (path / "requirements.txt").exists())
REQUIREMENTS_FILE = ROOT_DIR / "requirements.txt"
ROOT_DIR


PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler')

In [13]:
# Install every runtime dependency declared in requirements.txt into the current notebook kernel environment.
print(f"Installing notebook dependencies from {REQUIREMENTS_FILE}")
!{sys.executable} -m pip install -r "{REQUIREMENTS_FILE}"

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

SRC_DIR


Installing notebook dependencies from /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/requirements.txt
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.0
    Uninstalling psutil-5.9.0:
      Successfully uninstalled psutil-5.9.0


PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/src')

In [ ]:
# Fail fast if the runtime environment or Steam API is not ready.
import os
import shutil
import socket
import subprocess
import sys

import psutil
import requests
from dotenv import load_dotenv

load_dotenv(ROOT_DIR / ".env", override=False)

# Report the execution node before any expensive stage starts.
hostname = socket.gethostname()
available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
memory = psutil.virtual_memory()
print(f"Hostname: {hostname}")
print(f"Available CPUs: {available_cpus}")
print(f"Total RAM (GiB): {memory.total / (1024 ** 3):.2f}")
print(f"Available RAM (GiB): {memory.available / (1024 ** 3):.2f}")

# Detect NVIDIA GPU visibility when the notebook runs on a GPU node.
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    gpu_command = [
        nvidia_smi,
        "--query-gpu=index,name,driver_version,memory.total,compute_cap",
        "--format=csv,noheader",
    ]
    gpu_result = subprocess.run(gpu_command, capture_output=True, text=True, check=False)
    if gpu_result.returncode == 0 and gpu_result.stdout.strip():
        print("NVIDIA GPUs:")
        for line in gpu_result.stdout.strip().splitlines():
            print(f"  {line}")
    else:
        print("NVIDIA GPU check failed or no GPUs were reported by nvidia-smi.")
        if gpu_result.stderr.strip():
            print(gpu_result.stderr.strip())
else:
    print("NVIDIA GPU check: nvidia-smi not found on this system.")

STEAM_API_KEY = os.getenv("STEAM_API_KEY")
if not STEAM_API_KEY:
    raise RuntimeError("STEAM_API_KEY is missing. Set it in the environment or in steam-crawler/.env before running the pipeline cells.")

preflight_response = requests.get(
    "https://api.steampowered.com/IStoreService/GetAppList/v1/",
    params={"key": STEAM_API_KEY, "max_results": 1},
    timeout=30,
)
preflight_response.raise_for_status()
preflight_payload = preflight_response.json()
apps = preflight_payload.get("response", {}).get("apps", [])
if not isinstance(apps, list):
    raise RuntimeError(f"Steam API preflight returned an unexpected payload: {preflight_payload}")

print(f"Steam API preflight succeeded with {len(apps)} sample app(s).")

# Also verify the store appdetails endpoint with a lightweight request.
appdetails_response = requests.get(
    "https://store.steampowered.com/api/appdetails",
    params={"appids": 10, "cc": "us", "l": "english", "filters": "basic"},
    timeout=30,
)
appdetails_response.raise_for_status()
appdetails_payload = appdetails_response.json()
appdetails_entry = appdetails_payload.get("10", {}) if isinstance(appdetails_payload, dict) else {}
if not isinstance(appdetails_entry, dict) or not appdetails_entry.get("success"):
    raise RuntimeError(f"Steam appdetails preflight returned an unexpected payload: {appdetails_payload}")

print("Steam appdetails preflight succeeded for app 10.")


## Runtime profile

Set `RUN_MODE` to `"smoke"` for small-batch validation or `"full"` for the full cluster run.
The smoke profile intentionally limits page/app/game counts so you can validate the pipeline cheaply.


In [15]:
import os

from steam_crawler import Config, Pipeline

# Set STEAM_RUN_MODE=full in SLURM for the full crawl; smoke remains the local default.
RUN_MODE = os.getenv("STEAM_RUN_MODE", "smoke").strip().lower()

# Full-run defaults for the real dataset build.
full_config = dict(
    sample_size=10_000,
    min_recommendations=5_000,
    reviews_per_game=1_000,
    recent_quota=500,
    helpful_quota=500,
    random_seed=5242,
    request_timeout_sec=30.0,
    max_retries=5,
    base_backoff_sec=1.0,
    max_backoff_sec=60.0,
    app_list_page_size=50_000,
)

# Smoke-run overrides for fast validation on a small slice of the data.
smoke_config = dict(
    sample_size=5,
    min_recommendations=5_000,
    reviews_per_game=20,
    recent_quota=10,
    helpful_quota=10,
    random_seed=5242,
    request_timeout_sec=30.0,
    max_retries=5,
    base_backoff_sec=1.0,
    max_backoff_sec=60.0,
    app_list_page_size=25,
)

# Stage-level smoke limits so the notebook never crawls the full dataset during validation.
full_limits = dict(max_pages=None, max_apps=None, sample_size=None, max_games=None)
smoke_limits = dict(max_pages=1, max_apps=25, sample_size=5, max_games=2)

active_config = smoke_config if RUN_MODE == "smoke" else full_config
active_limits = smoke_limits if RUN_MODE == "smoke" else full_limits

settings = Config.from_env(ROOT_DIR, **active_config)
pipeline = Pipeline(settings)

active_limits


{'max_pages': 1, 'max_apps': 25, 'sample_size': 5, 'max_games': 2}

## Stage cells

Each stage defaults to cache reuse and obeys the active smoke/full limits from the profile cell above.


In [16]:
# Stage 1: fetch a page-limited or full app catalog depending on RUN_MODE.
stage_01_result = pipeline.run_stage_01(force_refresh=False, max_pages=active_limits["max_pages"])
stage_01_result


2026-04-10 02:42:07,345 | INFO | Reusing cached stage 01 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv
2026-04-10 02:42:07,353 | INFO | stage_01 summary | skipped=True | rows=25 | elapsed=0.01s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv


StageResult(stage_name='stage_01', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv'), rows_written=25, skipped=True, elapsed_seconds=0.00739291699937894, retry_count=0, error_count=0)

In [17]:
# Stage 2: fetch appdetails for a bounded number of apps in smoke mode.
stage_02_result = pipeline.run_stage_02(force_refresh=False, max_apps=active_limits["max_apps"])
stage_02_result


2026-04-10 02:42:07,383 | INFO | Reusing cached stage 02 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz
2026-04-10 02:42:07,388 | INFO | stage_02 summary | skipped=True | rows=25 | elapsed=0.01s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz


StageResult(stage_name='stage_02', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz'), rows_written=25, skipped=True, elapsed_seconds=0.011499915999593213, retry_count=0, error_count=0)

In [18]:
# Stage 3: merge cached app rows with cached appdetails rows.
stage_03_result = pipeline.run_stage_03(force_refresh=False, max_apps=active_limits["max_apps"])
stage_03_result


2026-04-10 02:42:07,406 | INFO | Reusing cached stage 03 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz
2026-04-10 02:42:07,408 | INFO | stage_03 summary | skipped=True | rows=25 | elapsed=0.00s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz


StageResult(stage_name='stage_03', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz'), rows_written=25, skipped=True, elapsed_seconds=0.002530625000872533, retry_count=0, error_count=0)

In [19]:
# Stage 4: sample games after metadata has been cached.
stage_04_result = pipeline.run_stage_04(force_refresh=False, sample_size=active_limits["sample_size"])
stage_04_result


2026-04-10 02:42:07,418 | INFO | Reusing cached stage 04 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04_selected_games.csv
2026-04-10 02:42:07,419 | INFO | stage_04 summary | skipped=True | rows=5 | elapsed=0.00s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04_selected_games.csv


StageResult(stage_name='stage_04', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04_selected_games.csv'), rows_written=5, skipped=True, elapsed_seconds=0.0015270000003511086, retry_count=0, error_count=0)

In [20]:
# Stage 5: pull reviews for a limited number of selected games in smoke mode.
stage_05_result = pipeline.run_stage_05(force_refresh=False, max_games=active_limits["max_games"])
stage_05_result


2026-04-10 02:42:07,428 | INFO | Reusing cached stage 05 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_reviews_dataset.csv.gz
2026-04-10 02:42:07,429 | INFO | stage_05 summary | skipped=True | rows=40 | elapsed=0.00s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_reviews_dataset.csv.gz


StageResult(stage_name='stage_05', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_reviews_dataset.csv.gz'), rows_written=40, skipped=True, elapsed_seconds=0.0022019159987394232, retry_count=0, error_count=0)

## One-shot execution

Run this cell when you want the notebook to execute all missing stages under the current smoke/full profile.


In [21]:
# Execute all missing stages under the currently selected profile.
all_stage_results = pipeline.run_all_missing(
    force_refresh=False,
    max_pages=active_limits["max_pages"],
    max_apps=active_limits["max_apps"],
    sample_size=active_limits["sample_size"],
    max_games=active_limits["max_games"],
)
all_stage_results


2026-04-10 02:42:07,437 | INFO | Reusing cached stage 01 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv
2026-04-10 02:42:07,438 | INFO | stage_01 summary | skipped=True | rows=25 | elapsed=0.00s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv
2026-04-10 02:42:07,441 | INFO | Reusing cached stage 02 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz
2026-04-10 02:42:07,442 | INFO | stage_02 summary | skipped=True | rows=25 | elapsed=0.00s | retries=0 | errors=0 | output=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz
2026-04-10 02:42:07,443 | INFO | Reusing cached stage 03 output: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz
2026-0

[StageResult(stage_name='stage_01', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv'), rows_written=25, skipped=True, elapsed_seconds=0.001019458000882878, retry_count=0, error_count=0),
 StageResult(stage_name='stage_02', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz'), rows_written=25, skipped=True, elapsed_seconds=0.0040425830011372454, retry_count=0, error_count=0),
 StageResult(stage_name='stage_03', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz'), rows_written=25, skipped=True, elapsed_seconds=0.0016017920006561326, retry_count=0, error_count=0),
 StageResult(stage_name='stage_04', output_path=PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04_selected_games.csv'), rows_